# Anchor extraction demo

Core engine: the LLM returns **verbatim boundary anchors only**; Python resolves offsets, slices text, groups segments by role, and stitches across chunks.

An **extraction profile** (detection prompt + optional unit-boundary regex) decides which logical units to extract. Bundled profiles: `anchor_extract/prompts/profiles/`.

Set `PDF_PATH` to a text-extractable PDF. The default points at the parent repo's AI RMF Playbook if present. Docs: `docs/core_pipeline.md`, `docs/profiles.md`.

In [1]:
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv

from anchor_extract import extract_pdf, build_anchor_system_prompt
    
from anchor_extract.pipeline import extract_document, to_requirements_json, save_json
from anchor_extract.settings import EXAMPLE_AI_RMF_BOUNDARY_PATTERN

load_dotenv()

# Profile: NIST AI RMF Playbook (primary example)
PDF_PATH = Path("../pdf/AI_RMF_Playbook.pdf")
DETECTION_PROMPT_PATH = Path("anchor_extract/prompts/profiles/ai_rmf_playbook.txt")
BOUNDARY_PATTERN = EXAMPLE_AI_RMF_BOUNDARY_PATTERN

# Alternative profile: HIPAA Administrative Simplification
# PDF_PATH = Path("notebooks/hipaa-simplification-201303.pdf")
# DETECTION_PROMPT_PATH = Path("anchor_extract/prompts/profiles/hipaa.txt")
# BOUNDARY_PATTERN = None

ModuleNotFoundError: No module named 'anchor_extract'

In [ ]:
START_PAGE = 5
END_PAGE = 10

doc = extract_pdf(str(PDF_PATH), start_page=START_PAGE, end_page=END_PAGE)

blocks_df = pd.DataFrame([{
    "page": b.page,
    "char_start": b.char_start,
    "char_end": b.char_end,
    "text_preview": b.text[:120],
} for b in doc.blocks[:15]])
blocks_df


In [ ]:
detection_prompt = DETECTION_PROMPT_PATH.read_text(encoding="utf-8")
system_prompt = build_anchor_system_prompt(detection_prompt)
print(system_prompt[:400], "...")

In [ ]:
extraction = extract_document(
    str(PDF_PATH),
    detection_prompt,
    doc=doc,
    start_page=START_PAGE,
    end_page=END_PAGE,
    requirement_boundary_pattern=BOUNDARY_PATTERN,
    verbose=True,
)

In [ ]:
results_df = pd.DataFrame([{
    "requirement_id": r.requirement_id,
    "status": r.status,
    "verbatim_match": r.verbatim_match,
    "end_resolved": r.end_resolved,
    "n_segments": r.n_segments,
    "n_segments_resolved": r.n_segments_resolved,
    "doc_offset_start": r.doc_offset_start,
    "doc_offset_end": r.doc_offset_end,
} for r in extraction.requirements])
results_df

In [ ]:
for r in extraction.requirements:
    if r.status != "complete" or not r.verbatim_match:
        continue
    print("===", r.requirement_id, "===")
    print("requirement:", r.requirement_text.strip()[:200], "...")
    print("context:", (r.context_text or "").strip()[:200], "...")
    print("questionnaire:", (r.questionnaire_text or "").strip()[:200], "...")

In [ ]:
for r in extraction.requirements:
    if r.n_segments != 1 or r.doc_offset_start < 0:
        print(r.requirement_id, "invariant: n/a (multi-span or unresolved)")
        continue
    ok = doc.full_text[r.doc_offset_start:r.doc_offset_end] == r.original_text
    print(r.requirement_id, "invariant:", "ok" if ok else "BAD")

In [ ]:
out = Path("../outputs/requirements.json")
save_json(to_requirements_json("AI_RMF_Playbook", doc, extraction), out)
print("Wrote", out.resolve())